In [ ]:
#%pip install librosa music21 numpy matplotlib soundfile

In [27]:
import librosa
import numpy as np

In [30]:
'''
true pitches: 44, 50, 56
'''
import json
with open("nsynth-train/examples.json", "r") as f:
    data=json.load(f)

samples=[]
true_pitches=[]
true_notes=[]
for key,value in data.items():
    if "keyboard_acoustic" in key:
        if value["pitch"] in [44,50,56]:
            samples.append(str(key))
            true_pitches.append(value["pitch"])
            true_notes.append(librosa.midi_to_note(value['pitch']))


In [32]:
import os,shutil
os.makedirs("./input", exist_ok=True)

json_path = "nsynth-train/examples.json"
audio_folder = "nsynth-train/audio"
input_folder = "./input"

for file in samples:
    src=os.path.join(audio_folder, file + ".wav")
    dst=os.path.join(input_folder, file + ".wav")

    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print(f"Missing file: {file}")


In [42]:
import pandas as pd
json_path = "nsynth-train/examples.json"
input = "./input"

pred_pitches=[]
pred_notes=[]
for file in os.listdir(input):
    path = os.path.join(input,file)

    y,sr=librosa.load(path,sr=16000)

    f0 = librosa.yin(y, fmin=50, fmax=500, sr=sr)
    valid = f0[~np.isnan(f0)]
    pred_pitch = np.median(valid)

    pred_pitches.append(pred_pitch)
    pred_notes.append(librosa.hz_to_note(pred_pitch))

res=[]
for i in range(len(samples)):
    res.append([
        samples[i],
        true_pitches[i],
        true_notes[i],
        pred_pitches[i],
        pred_notes[i]
    ])

df=pd.DataFrame(
    res, columns=[
        "sample",
        "true_pitch",
        "true_note",
        "pred_pitch",
        "pred_note"
    ]
)

print(df)


                            sample  true_pitch true_note  pred_pitch pred_note
0    keyboard_acoustic_003-050-100          50        D3  207.877498       G♯3
1    keyboard_acoustic_011-050-050          50        D3  146.692939        D3
2    keyboard_acoustic_013-056-025          56       G♯3  104.066910       G♯2
3    keyboard_acoustic_014-044-025          44       G♯2  146.849344        D3
4    keyboard_acoustic_005-044-127          44       G♯2  104.192233       G♯2
..                             ...         ...       ...         ...       ...
278  keyboard_acoustic_015-056-050          56       G♯3  103.953737       G♯2
279  keyboard_acoustic_010-050-100          50        D3  208.153563       G♯3
280  keyboard_acoustic_014-044-050          44       G♯2  104.064915       G♯2
281  keyboard_acoustic_008-050-025          50        D3  111.857390        A2
282  keyboard_acoustic_007-056-025          56       G♯3  146.750807        D3

[283 rows x 5 columns]


In [57]:
df["true_note"] = df["true_note"].str.replace("♯", "#")
df["pred_note"] = df["pred_note"].str.replace("♯", "#")
df["true_note"] = df["true_note"].str.replace("♭", "b")
df["pred_note"] = df["pred_note"].str.replace("♭", "b")
df["Match"] = df["true_note"] == df["pred_note"]
accuracy = df["Match"].mean()
print(f"Accuracy: {accuracy:.2%}")
df.to_csv("results.csv", index=False)

Accuracy: 27.92%


In [ ]:
%pip install music21

In [59]:
from music21 import stream, note

s = stream.Stream()

for n_name in df["pred_note"]:
    n = note.Note(n_name)
    n.quarterLength = 1
    s.append(n)

s.write('musicxml', fp='./output/batch_output.xml')

PosixPath('/Users/amishasingh/Documents/sheetgenie/nsynth/output/batch_output.xml')

In [61]:
len(samples)

283